# PhishCross Notebook

In [ ]:
# ===========================================================
# CELL 1: SETUP ENVIRONMENT & PATH
# ===========================================================
import os, sys

# Jalur kode PhishCross dan data Phish360
CODE_PATH = "/kaggle/input/datasets/afrizalilzamunadhif/running-kaggle"
DATA_ROOT = "/kaggle/input/datasets/afrizalilzamunadhif/phish360/Phish360"

sys.path.insert(0, CODE_PATH)
os.environ["PYTHONPATH"] = CODE_PATH

WORK = "/kaggle/working/phishcross_runs"
for d in [WORK, f"{WORK}/features", f"{WORK}/results", f"{WORK}/checkpoints"]:
    os.makedirs(d, exist_ok=True)

from phishcross.config import Config
print("✅ phishcross OK")
print("✅ Phish360 OK" if os.path.isdir(DATA_ROOT) else f"❌ Phish360 TIDAK ditemukan di {DATA_ROOT}")

In [ ]:
# ===========================================================
# CELL 2: GENERATE MANIFEST
# ===========================================================
!PYTHONPATH={CODE_PATH} python -m phishcross.data.manifest --data-root {DATA_ROOT} --out-csv /kaggle/working/phishcross_runs/manifest.csv

import pandas as pd
df = pd.read_csv("/kaggle/working/phishcross_runs/manifest.csv")
print(df.groupby(["split", "label"]).size())
print(f"Total dataset: {len(df)}")

In [ ]:
# ===========================================================
# CELL 3: EXTRACT FEATURES (HTML dengan Head + Tail)
# ===========================================================
# Fitur HTML dipotong menggunakan metode Head + Tail (2000 depan + 2000 belakang)
!PYTHONPATH={CODE_PATH} python -m phishcross.extract_features --manifest /kaggle/working/phishcross_runs/manifest.csv --feature-dir /kaggle/working/phishcross_runs/features --data-root {DATA_ROOT} --batch-size 64 --skip-existing

import os
for f in sorted(os.listdir("/kaggle/working/phishcross_runs/features")):
    sz = os.path.getsize(f"/kaggle/working/phishcross_runs/features/{f}") / 1024 / 1024
    print(f"  ✅ {f} ({sz:.1f} MB)")

In [ ]:
# ===========================================================
# CELL 4: JALANKAN SEMUA EKSPERIMEN (Multi-Seed & Ablation)
# ===========================================================
# Orchestrator run_all_experiments.py.
# Parameter:
#   --seeds 42,43,44 : Menjalankan 3 seed berbeda untuk validasi statistik.
#   --extra : Menjalankan eksperimen missing-modality dan baseline ablation no_moddrop.
#   --epochs 30 : Jumlah epoch maksimum (Early stopping akan menghentikan jika sudah konvergen).
#   --skip-features : Karena sudah mengekstrak fitur di Cell 3.

!PYTHONPATH={CODE_PATH} python {CODE_PATH}/run_all_experiments.py \
    --data-root {DATA_ROOT} \
    --work-dir /kaggle/working/phishcross_runs \
    --epochs 30 \
    --batch-size 64 \
    --seeds 42,43,44 \
    --extra \
    --skip-features

In [ ]:
# ===========================================================
# CELL 5: RINGKASAN & ANALISIS HASIL
# ===========================================================
import pandas as pd
from pathlib import Path

result_dir = Path("/kaggle/working/phishcross_runs/results")
summary_file = result_dir / "summary.csv"
all_runs_file = result_dir / "all_runs.csv"

if all_runs_file.exists():
    print("=== SEMUA RUNS (PER SEED) ===")
    df_raw = pd.read_csv(all_runs_file)
    print(df_raw.to_string(index=False))
    
if summary_file.exists():
    print("\n=== RINGKASAN AGREGASI (MEAN ± STD) ===")
    df_summary = pd.read_csv(summary_file)
    print(df_summary.to_string(index=False))

In [ ]:
# ===========================================================
# CELL 6: DOWNLOAD SEMUA HASIL SEBAGAI ZIP
# ===========================================================
import zipfile
from pathlib import Path

WORK = Path("/kaggle/working/phishcross_runs")
ZIP_PATH = "/kaggle/working/phishcross_result.zip"

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(WORK.rglob("*")):
        if f.is_file():
            arcname = f"phishcross_runs/{f.relative_to(WORK)}"
            size_mb = f.stat().st_size / 1024 / 1024
            print(f"  📦 {arcname} ({size_mb:.1f} MB)")
            zf.write(f, arcname)

zip_size = Path(ZIP_PATH).stat().st_size / 1024 / 1024
print(f"\n✅ ZIP berhasil dibuat: {ZIP_PATH}")
print(f"📁 Ukuran: {zip_size:.1f} MB")
print("\n👉 Buka tab 'Output' di kanan -> klik 'phishcross_result.zip' -> Download")